<a href="https://colab.research.google.com/github/Praharshita1275/python/blob/main/FinalColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

All 3 modals

In [ ]:
# Install OpenAI Whisper
!pip install git+https://github.com/openai/whisper.git

# Install FFmpeg (Required for audio processing)
!sudo apt update && sudo apt install ffmpeg

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-kpdyr487
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-kpdyr487
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=6893e0308e435c9a7199b92e0a5c5a79a7b502c78311a7f601dcdc8bd832117a
  Stored in directory: /tmp/pip-ephem-wheel-cache-shp2743t/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 http://security.ubuntu.com/u

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from transformers import pipeline, CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util
import warnings
import pandas as pd
import torch
import json
import whisper
from PIL import Image
import os

# --- Initial Setup ---
warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', None)

# ==========================================
# 1. LOAD MODELS (Run Once Global)
# ==========================================
print("⏳ Initializing Global Models (this happens once)...")

# A. NLP Models
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = ["criminal conspiracy", "issuing a command", "normal conversation"]
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# B. Audio Model (Whisper)
print("   - Loading Whisper...")
audio_model = whisper.load_model("base")

# C. Image Model (CLIP)
print("   - Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# D. Define Knowledge Base
crime_image_labels = [
    "a pile of cash money", "a gun or firearm", "drugs or white powder",
    "a secret map or blueprint", "a black duffel bag", "a police car",
    "a normal photo of a person", "a landscape", "a cat or dog"
]

prototypes_by_category = {
    "Threats & Enforcement": [ "Make him understand the consequences.", "Send a clear message.", "Neutralize the threat." ],
    "Financial Transactions": [ "Confirm the payment.", "Launder the money.", "Settle the accounts.", "Look at this cash." ],
    "Commands & Directives": [ "Get the job done now.", "Proceed with the plan.", "Green-light the operation." ],
    "Secrecy & Evasion": [ "Use the secure line.", "Delete this after reading.", "Use the burner phone." ],
    "Acquiring Resources": [ "Acquire the equipment.", "Source the vehicle.", "Here are the weapons." ],
    "Logistics & Transport": [ "Coordinate the pickup.", "The package is delivered.", "Move the merchandise." ],
    "Reporting & Status Updates": [ "Report your status.", "Target is under surveillance.", "Operation complete." ]
}

category_weights = {
    "Commands & Directives": 1.8, "Financial Transactions": 1.6, "Threats & Enforcement": 1.5,
    "Secrecy & Evasion": 1.2, "Logistics & Transport": 0.8, "Acquiring Resources": 0.7, "Reporting & Status Updates": 0.5
}

# Pre-compute prototype embeddings
prototype_embeddings_by_category = {}
for category, sentences in prototypes_by_category.items():
    prototype_embeddings_by_category[category] = embedding_model.encode(sentences, convert_to_tensor=True)

print("✅ Models Loaded.\n")


# ==========================================
# 2. PROCESSING FUNCTION
# ==========================================
def process_criminal_dataset(dataset_num, active_modalities):
    """
    Args:
        dataset_num (int): Number of the dataset (1-8).
        active_modalities (list): List containing strings like ['text', 'audio', 'image'].
    Returns:
        float: The calculated System Confidence Score.
    """

    # 1. Setup Dynamic Paths
    base_path = "/content/drive/MyDrive/Datasets/New Datasets"
    input_file = f"{base_path}/Data/data{dataset_num}_multimodal_final.json"
    output_graph_img = f"{base_path}/Final_Outputs/data{dataset_num}_multimodal.png"
    output_scores_csv = f"{base_path}/Final_Outputs/data{dataset_num}_multimodal.csv"

    print(f"\n--- Processing Dataset {dataset_num} ---")
    print(f"📂 Loading: {input_file}")

    try:
        with open(input_file, 'r') as f:
            message_data = json.load(f)
    except Exception as e:
        print(f"❌ Failed to load dataset {dataset_num}: {e}")
        return 0.0

    directed_edge_weights = {}

    # 2. Analyze Messages
    for msg in message_data:
        sender = msg['sender']
        receiver = msg['receiver']

        text = msg.get('message_text')
        audio_path = msg.get('audio_path')
        image_path = msg.get('image_path')

        # --- A. Process Audio (Only if 'audio' is requested) ---
        if 'audio' in active_modalities and (not text) and audio_path:
            try:
                transcription = audio_model.transcribe(audio_path)
                text = transcription["text"]
            except: pass

        # --- B. Process Image (Only if 'image' is requested) ---
        if 'image' in active_modalities and (not text) and image_path:
            try:
                image = Image.open(image_path)
                inputs = clip_processor(text=crime_image_labels, images=image, return_tensors="pt", padding=True)
                outputs = clip_model(**inputs)
                best_idx = outputs.logits_per_image.softmax(dim=1).argmax().item()
                text = f"User sent a photo of {crime_image_labels[best_idx]}"
            except: pass

        # --- C. Process Text (Only if 'text' is requested) ---
        # Note: We rely on 'text' being present or generated above.
        # If user turned off text analysis for raw text messages, we skip this block.
        if not text: continue
        if 'text' not in active_modalities and not (audio_path or image_path):
            # If it was a raw text message but 'text' modality is OFF
            continue

        # --- Scoring Logic ---
        try:
            # Zero-shot
            result = classifier(text, candidate_labels, multi_label=False)
            score_map = {label: score for label, score in zip(result['labels'], result['scores'])}
            c_score = (score_map.get("criminal conspiracy", 0) * 1.5) + (score_map.get("issuing a command", 0) * 1.0)

            # Semantic Similarity
            msg_emb = embedding_model.encode(text, convert_to_tensor=True)
            sem_score = 0
            for cat, proto_embs in prototype_embeddings_by_category.items():
                best_sim = torch.max(util.cos_sim(msg_emb, proto_embs)).item()
                curr_score = best_sim * category_weights[cat]
                if curr_score > sem_score: sem_score = curr_score

            # Hybrid Score
            final_weight = ((0.3 * c_score) + (0.7 * sem_score)) * 2.0

            edge = (sender, receiver)
            directed_edge_weights[edge] = directed_edge_weights.get(edge, 0) + final_weight

        except: continue

    # 3. Build Graph & HITS
    G = nx.DiGraph()
    for (u, v), w in directed_edge_weights.items():
        if w > 1.2: G.add_edge(u, v, weight=round(w, 2))

    if G.number_of_nodes() == 0:
        print("   ⚠️ Graph empty (no strong evidence found).")
        return 0.0

    hubs, authorities = nx.hits(G, max_iter=1000, normalized=True)

    # 4. Identify Mastermind
    avg_hub = sum(hubs.values()) / len(hubs) if hubs else 0
    potential_mms = [n for n in G.nodes() if hubs[n] > avg_hub * 1.2]
    mastermind = max(potential_mms, key=lambda x: hubs[x]) if potential_mms else None

    # 5. Identify Roles
    avg_auth = sum(authorities.values()) / len(authorities) if authorities else 0
    roles = {}
    for n in G.nodes():
        if n == mastermind: roles[n] = "Mastermind"
        elif authorities[n] > avg_auth * 1.5: roles[n] = "Middleman"
        else: roles[n] = "Follower"

    # 6. Calculate Confidence
    # MM Confidence
    mm_conf = 0.0
    if mastermind:
        mm_hub = hubs[mastermind]
        others = [h for n, h in hubs.items() if n != mastermind]
        runner_up = max(others) if others else 0
        if mm_hub > 0: mm_conf = (mm_hub - runner_up) / mm_hub

    # Mid Confidence
    mid_conf = 0.0
    mids = [n for n, r in roles.items() if r == "Middleman"]
    fols = [n for n, r in roles.items() if r == "Follower"]
    if mids and fols:
        avg_mid = sum(authorities[n] for n in mids) / len(mids)
        avg_fol = sum(authorities[n] for n in fols) / len(fols)
        if avg_mid > 0: mid_conf = min((avg_mid - avg_fol) / avg_mid, 1.0)

    system_confidence = (0.6 * mm_conf) + (0.4 * mid_conf)

    # 7. Save Outputs
    # Save CSV
    df = pd.DataFrame({'Hub': hubs, 'Auth': authorities, 'Role': roles})
    df.to_csv(output_scores_csv)

    # Save Image
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)
    colors = [{"Mastermind": "red", "Middleman": "orange", "Follower": "skyblue"}.get(roles[n], "gray") for n in G.nodes()]
    nx.draw(G, pos, with_labels=True, node_color=colors, node_size=2500, font_weight='bold')
    plt.savefig(output_graph_img)
    plt.close() # Close plot to save memory

    print(f"   ✅ Saved: Graph & CSV. | Confidence: {system_confidence:.2%}")
    return system_confidence


# ==========================================
# 3. MAIN EXECUTION LOOP
# ==========================================

# --- CONFIGURATION ---
# Select modalities here: ['text', 'audio', 'image']
# You can perform combinations like ['image', 'audio'] or just ['text']
selected_modalities = ['text', 'audio', 'image']

# Range of datasets (1 to 8)
dataset_range = range(1, 9)
# ---------------------

print("\n" + "="*40)
print(f"🚀 STARTING BATCH PROCESS")
print(f"   Modalities Active: {selected_modalities}")
print("="*40)

results_summary = []

for i in dataset_range:
    score = process_criminal_dataset(i, selected_modalities)
    results_summary.append({
        "Dataset": f"Data {i}",
        "Modality": "+".join(selected_modalities),
        "Confidence Score": f"{score:.2%}"
    })

# --- Final Output ---
print("\n" + "="*40)
print("📊 FINAL BATCH ACCURACY REPORT")
print("="*40)
results_df = pd.DataFrame(results_summary)
print(results_df)
print("="*40)

⏳ Initializing Global Models (this happens once)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ Models Loaded.


🚀 STARTING BATCH PROCESS
   Modalities Active: ['text', 'audio', 'image']

--- Processing Dataset 1 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data1_multimodal_final.json


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


   ✅ Saved: Graph & CSV. | Confidence: 74.83%

--- Processing Dataset 2 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data2_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 78.31%

--- Processing Dataset 3 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data3_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 87.73%

--- Processing Dataset 4 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data4_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 82.65%

--- Processing Dataset 5 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data5_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 73.65%

--- Processing Dataset 6 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data6_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 80.78%

--- Processing Dataset 7 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data7_multimodal_final.json
   ✅ Saved

Visualization modified


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from transformers import pipeline, CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util
import warnings
import pandas as pd
import torch
import json
import whisper
from PIL import Image
import os

# --- Initial Setup ---
warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', None)

# ==========================================
# 1. LOAD MODELS (Run Once Global)
# ==========================================
print("⏳ Initializing Global Models (this happens once)...")

# A. NLP Models
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = ["criminal conspiracy", "issuing a command", "normal conversation"]
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# B. Audio Model (Whisper)
print("   - Loading Whisper...")
audio_model = whisper.load_model("base")

# C. Image Model (CLIP)
print("   - Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# D. Define Knowledge Base
crime_image_labels = [
    "a pile of cash money", "a gun or firearm", "drugs or white powder",
    "a secret map or blueprint", "a black duffel bag", "a police car",
    "a normal photo of a person", "a landscape", "a cat or dog"
]

prototypes_by_category = {
    "Threats & Enforcement": [ "Make him understand the consequences.", "Send a clear message.", "Neutralize the threat." ],
    "Financial Transactions": [ "Confirm the payment.", "Launder the money.", "Settle the accounts.", "Look at this cash." ],
    "Commands & Directives": [ "Get the job done now.", "Proceed with the plan.", "Green-light the operation." ],
    "Secrecy & Evasion": [ "Use the secure line.", "Delete this after reading.", "Use the burner phone." ],
    "Acquiring Resources": [ "Acquire the equipment.", "Source the vehicle.", "Here are the weapons." ],
    "Logistics & Transport": [ "Coordinate the pickup.", "The package is delivered.", "Move the merchandise." ],
    "Reporting & Status Updates": [ "Report your status.", "Target is under surveillance.", "Operation complete." ]
}

category_weights = {
    "Commands & Directives": 1.8, "Financial Transactions": 1.6, "Threats & Enforcement": 1.5,
    "Secrecy & Evasion": 1.2, "Logistics & Transport": 0.8, "Acquiring Resources": 0.7, "Reporting & Status Updates": 0.5
}

# Pre-compute prototype embeddings
prototype_embeddings_by_category = {}
for category, sentences in prototypes_by_category.items():
    prototype_embeddings_by_category[category] = embedding_model.encode(sentences, convert_to_tensor=True)

print("✅ Models Loaded.\n")


# ==========================================
# 2. PROCESSING FUNCTION
# ==========================================
def process_criminal_dataset(dataset_num, active_modalities):
    """
    Args:
        dataset_num (int): Number of the dataset (1-8).
        active_modalities (list): List containing strings like ['text', 'audio', 'image'].
    Returns:
        float: The calculated System Confidence Score.
    """

    # 1. Setup Dynamic Paths
    base_path = "/content/drive/MyDrive/Datasets/New Datasets"
    input_file = f"{base_path}/Data/data{dataset_num}_multimodal_final.json"
    output_graph_img = f"{base_path}/Final_Outputs/data{dataset_num}_multimodal.png"
    output_scores_csv = f"{base_path}/Final_Outputs/data{dataset_num}_multimodal.csv"

    print(f"\n--- Processing Dataset {dataset_num} ---")
    print(f"📂 Loading: {input_file}")

    try:
        with open(input_file, 'r') as f:
            message_data = json.load(f)
    except Exception as e:
        print(f"❌ Failed to load dataset {dataset_num}: {e}")
        return 0.0

    directed_edge_weights = {}

    # 2. Analyze Messages
    for msg in message_data:
        sender = msg['sender']
        receiver = msg['receiver']

        text = msg.get('message_text')
        audio_path = msg.get('audio_path')
        image_path = msg.get('image_path')

        # --- A. Process Audio (Only if 'audio' is requested) ---
        if 'audio' in active_modalities and (not text) and audio_path:
            try:
                transcription = audio_model.transcribe(audio_path)
                text = transcription["text"]
            except: pass

        # --- B. Process Image (Only if 'image' is requested) ---
        if 'image' in active_modalities and (not text) and image_path:
            try:
                image = Image.open(image_path)
                inputs = clip_processor(text=crime_image_labels, images=image, return_tensors="pt", padding=True)
                outputs = clip_model(**inputs)
                best_idx = outputs.logits_per_image.softmax(dim=1).argmax().item()
                text = f"User sent a photo of {crime_image_labels[best_idx]}"
            except: pass

        # --- C. Process Text (Only if 'text' is requested) ---
        # Note: We rely on 'text' being present or generated above.
        # If user turned off text analysis for raw text messages, we skip this block.
        if not text: continue
        if 'text' not in active_modalities and not (audio_path or image_path):
            # If it was a raw text message but 'text' modality is OFF
            continue

        # --- Scoring Logic ---
        try:
            # Zero-shot
            result = classifier(text, candidate_labels, multi_label=False)
            score_map = {label: score for label, score in zip(result['labels'], result['scores'])}
            c_score = (score_map.get("criminal conspiracy", 0) * 1.5) + (score_map.get("issuing a command", 0) * 1.0)

            # Semantic Similarity
            msg_emb = embedding_model.encode(text, convert_to_tensor=True)
            sem_score = 0
            for cat, proto_embs in prototype_embeddings_by_category.items():
                best_sim = torch.max(util.cos_sim(msg_emb, proto_embs)).item()
                curr_score = best_sim * category_weights[cat]
                if curr_score > sem_score: sem_score = curr_score

            # Hybrid Score
            final_weight = ((0.3 * c_score) + (0.7 * sem_score)) * 2.0

            edge = (sender, receiver)
            directed_edge_weights[edge] = directed_edge_weights.get(edge, 0) + final_weight

        except: continue

    # 3. Build Graph & HITS
    G = nx.DiGraph()
    for (u, v), w in directed_edge_weights.items():
        if w > 1.2: G.add_edge(u, v, weight=round(w, 2))

    if G.number_of_nodes() == 0:
        print("   ⚠️ Graph empty (no strong evidence found).")
        return 0.0

    hubs, authorities = nx.hits(G, max_iter=1000, normalized=True)

    # 4. Identify Mastermind
    avg_hub = sum(hubs.values()) / len(hubs) if hubs else 0
    potential_mms = [n for n in G.nodes() if hubs[n] > avg_hub * 1.2]
    mastermind = max(potential_mms, key=lambda x: hubs[x]) if potential_mms else None

    # 5. Identify Roles
    avg_auth = sum(authorities.values()) / len(authorities) if authorities else 0
    roles = {}
    for n in G.nodes():
        if n == mastermind: roles[n] = "Mastermind"
        elif authorities[n] > avg_auth * 1.5: roles[n] = "Middleman"
        else: roles[n] = "Follower"

    # 6. Calculate Confidence
    # MM Confidence
    mm_conf = 0.0
    if mastermind:
        mm_hub = hubs[mastermind]
        others = [h for n, h in hubs.items() if n != mastermind]
        runner_up = max(others) if others else 0
        if mm_hub > 0: mm_conf = (mm_hub - runner_up) / mm_hub

    # Mid Confidence
    mid_conf = 0.0
    mids = [n for n, r in roles.items() if r == "Middleman"]
    fols = [n for n, r in roles.items() if r == "Follower"]
    if mids and fols:
        avg_mid = sum(authorities[n] for n in mids) / len(mids)
        avg_fol = sum(authorities[n] for n in fols) / len(fols)
        if avg_mid > 0: mid_conf = min((avg_mid - avg_fol) / avg_mid, 1.0)

    system_confidence = (0.6 * mm_conf) + (0.4 * mid_conf)

    # 7. Save Outputs
    # 7. Save Outputs
    # Save CSV
    df = pd.DataFrame({'Hub': hubs, 'Auth': authorities, 'Role': roles})
    df.to_csv(output_scores_csv)

    # --- UPDATED VISUALIZATION BLOCK ---
    if G.number_of_nodes() > 0:
        # Use spring layout with k parameter for better spacing
        pos = nx.spring_layout(G, k=2.5, seed=42, iterations=80)

        # Define colors based on roles
        role_colors_map = { "Mastermind": "red", "Middleman": "orange", "Follower": "skyblue" }
        node_colors = [role_colors_map.get(roles.get(node, "Follower"), "skyblue") for node in G.nodes()]
        edge_weights = [G[u][v]['weight'] for u, v in G.edges()]

        plt.figure(figsize=(16, 12))

        # Draw Nodes
        nx.draw_networkx_nodes(G, pos, node_size=3000, node_color=node_colors, edgecolors='black')

        # Draw Edges (width based on weight)
        nx.draw_networkx_edges(G, pos, width=[w * 2 for w in edge_weights],
                               edge_color='gray', arrowsize=20, node_size=3000)

        # Draw Labels
        nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')

        # Draw Edge Weights
        edge_labels = nx.get_edge_attributes(G, 'weight')
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='darkgreen')

        # Custom Legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', label='Mastermind (Top Hub)', markerfacecolor='red', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Middleman (High Authority)', markerfacecolor='orange', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Follower', markerfacecolor='skyblue', markersize=15)
        ]
        plt.legend(handles=legend_elements, title="Roles", loc="upper right")

        plt.title(f"Influence Network - Dataset {dataset_num}", size=20)
        plt.axis('off')

        # Save to the dynamic path defined earlier
        plt.savefig(output_graph_img)
        plt.close() # Important: Close plot to free memory in loop
    else:
        print("   ⚠️ Graph is empty, skipping visualization.")

    print(f"   ✅ Saved: Graph & CSV. | Confidence: {system_confidence:.2%}")
    return system_confidence


# ==========================================
# 3. MAIN EXECUTION LOOP
# ==========================================

# --- CONFIGURATION ---
# Select modalities here: ['text', 'audio', 'image']
# You can perform combinations like ['image', 'audio'] or just ['text']
selected_modalities = ['text', 'audio', 'image']

# Range of datasets (1 to 8)
dataset_range = range(1, 9)
# ---------------------

print("\n" + "="*40)
print(f"🚀 STARTING BATCH PROCESS")
print(f"   Modalities Active: {selected_modalities}")
print("="*40)

results_summary = []

for i in dataset_range:
    score = process_criminal_dataset(i, selected_modalities)
    results_summary.append({
        "Dataset": f"Data {i}",
        "Modality": "+".join(selected_modalities),
        "Confidence Score": f"{score:.2%}"
    })

# --- Final Output ---
print("\n" + "="*40)
print("📊 FINAL BATCH ACCURACY REPORT")
print("="*40)
results_df = pd.DataFrame(results_summary)
print(results_df)
print("="*40)

⏳ Initializing Global Models (this happens once)...


Device set to use cuda:0


   - Loading Whisper...
   - Loading CLIP...
✅ Models Loaded.


🚀 STARTING BATCH PROCESS
   Modalities Active: ['text', 'audio', 'image']

--- Processing Dataset 1 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data1_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 74.83%

--- Processing Dataset 2 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data2_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 78.31%

--- Processing Dataset 3 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data3_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 87.73%

--- Processing Dataset 4 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data4_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 82.65%

--- Processing Dataset 5 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data5_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 73.65%

--- Processing Dataset 6 ---
📂 Loadi

Text+Audio

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from transformers import pipeline, CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util
import warnings
import pandas as pd
import torch
import json
import whisper
from PIL import Image
import os

# --- Initial Setup ---
warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', None)

# ==========================================
# 1. LOAD MODELS (Run Once Global)
# ==========================================
print("⏳ Initializing Global Models (this happens once)...")

# A. NLP Models
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = ["criminal conspiracy", "issuing a command", "normal conversation"]
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# B. Audio Model (Whisper)
print("   - Loading Whisper...")
audio_model = whisper.load_model("base")

# C. Image Model (CLIP)
print("   - Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# D. Define Knowledge Base
crime_image_labels = [
    "a pile of cash money", "a gun or firearm", "drugs or white powder",
    "a secret map or blueprint", "a black duffel bag", "a police car",
    "a normal photo of a person", "a landscape", "a cat or dog"
]

prototypes_by_category = {
    "Threats & Enforcement": [ "Make him understand the consequences.", "Send a clear message.", "Neutralize the threat." ],
    "Financial Transactions": [ "Confirm the payment.", "Launder the money.", "Settle the accounts.", "Look at this cash." ],
    "Commands & Directives": [ "Get the job done now.", "Proceed with the plan.", "Green-light the operation." ],
    "Secrecy & Evasion": [ "Use the secure line.", "Delete this after reading.", "Use the burner phone." ],
    "Acquiring Resources": [ "Acquire the equipment.", "Source the vehicle.", "Here are the weapons." ],
    "Logistics & Transport": [ "Coordinate the pickup.", "The package is delivered.", "Move the merchandise." ],
    "Reporting & Status Updates": [ "Report your status.", "Target is under surveillance.", "Operation complete." ]
}

category_weights = {
    "Commands & Directives": 1.8, "Financial Transactions": 1.6, "Threats & Enforcement": 1.5,
    "Secrecy & Evasion": 1.2, "Logistics & Transport": 0.8, "Acquiring Resources": 0.7, "Reporting & Status Updates": 0.5
}

# Pre-compute prototype embeddings
prototype_embeddings_by_category = {}
for category, sentences in prototypes_by_category.items():
    prototype_embeddings_by_category[category] = embedding_model.encode(sentences, convert_to_tensor=True)

print("✅ Models Loaded.\n")


# ==========================================
# 2. PROCESSING FUNCTION
# ==========================================
def process_criminal_dataset(dataset_num, active_modalities):
    """
    Args:
        dataset_num (int): Number of the dataset (1-8).
        active_modalities (list): List containing strings like ['text', 'audio', 'image'].
    Returns:
        float: The calculated System Confidence Score.
    """

    # 1. Setup Dynamic Paths
    base_path = "/content/drive/MyDrive/Datasets/New Datasets"
    input_file = f"{base_path}/Data/data{dataset_num}_multimodal_final.json"
    output_graph_img = f"{base_path}/Text_Audio_Outputs/data{dataset_num}_multimodal.png"
    output_scores_csv = f"{base_path}/Text_Audio_Outputs/data{dataset_num}_multimodal.csv"

    print(f"\n--- Processing Dataset {dataset_num} ---")
    print(f"📂 Loading: {input_file}")

    try:
        with open(input_file, 'r') as f:
            message_data = json.load(f)
    except Exception as e:
        print(f"❌ Failed to load dataset {dataset_num}: {e}")
        return 0.0

    directed_edge_weights = {}

    # 2. Analyze Messages
    for msg in message_data:
        sender = msg['sender']
        receiver = msg['receiver']

        text = msg.get('message_text')
        audio_path = msg.get('audio_path')
        image_path = msg.get('image_path')

        # --- A. Process Audio (Only if 'audio' is requested) ---
        if 'audio' in active_modalities and (not text) and audio_path:
            try:
                transcription = audio_model.transcribe(audio_path)
                text = transcription["text"]
            except: pass

        # --- B. Process Image (Only if 'image' is requested) ---
        if 'image' in active_modalities and (not text) and image_path:
            try:
                image = Image.open(image_path)
                inputs = clip_processor(text=crime_image_labels, images=image, return_tensors="pt", padding=True)
                outputs = clip_model(**inputs)
                best_idx = outputs.logits_per_image.softmax(dim=1).argmax().item()
                text = f"User sent a photo of {crime_image_labels[best_idx]}"
            except: pass

        # --- C. Process Text (Only if 'text' is requested) ---
        # Note: We rely on 'text' being present or generated above.
        # If user turned off text analysis for raw text messages, we skip this block.
        if not text: continue
        if 'text' not in active_modalities and not (audio_path or image_path):
            # If it was a raw text message but 'text' modality is OFF
            continue

        # --- Scoring Logic ---
        try:
            # Zero-shot
            result = classifier(text, candidate_labels, multi_label=False)
            score_map = {label: score for label, score in zip(result['labels'], result['scores'])}
            c_score = (score_map.get("criminal conspiracy", 0) * 1.5) + (score_map.get("issuing a command", 0) * 1.0)

            # Semantic Similarity
            msg_emb = embedding_model.encode(text, convert_to_tensor=True)
            sem_score = 0
            for cat, proto_embs in prototype_embeddings_by_category.items():
                best_sim = torch.max(util.cos_sim(msg_emb, proto_embs)).item()
                curr_score = best_sim * category_weights[cat]
                if curr_score > sem_score: sem_score = curr_score

            # Hybrid Score
            final_weight = ((0.3 * c_score) + (0.7 * sem_score)) * 2.0

            edge = (sender, receiver)
            directed_edge_weights[edge] = directed_edge_weights.get(edge, 0) + final_weight

        except: continue

    # 3. Build Graph & HITS
    G = nx.DiGraph()
    for (u, v), w in directed_edge_weights.items():
        if w > 1.2: G.add_edge(u, v, weight=round(w, 2))

    if G.number_of_nodes() == 0:
        print("   ⚠️ Graph empty (no strong evidence found).")
        return 0.0

    hubs, authorities = nx.hits(G, max_iter=1000, normalized=True)

    # 4. Identify Mastermind
    avg_hub = sum(hubs.values()) / len(hubs) if hubs else 0
    potential_mms = [n for n in G.nodes() if hubs[n] > avg_hub * 1.2]
    mastermind = max(potential_mms, key=lambda x: hubs[x]) if potential_mms else None

    # 5. Identify Roles
    avg_auth = sum(authorities.values()) / len(authorities) if authorities else 0
    roles = {}
    for n in G.nodes():
        if n == mastermind: roles[n] = "Mastermind"
        elif authorities[n] > avg_auth * 1.5: roles[n] = "Middleman"
        else: roles[n] = "Follower"

    # 6. Calculate Confidence
    # MM Confidence
    mm_conf = 0.0
    if mastermind:
        mm_hub = hubs[mastermind]
        others = [h for n, h in hubs.items() if n != mastermind]
        runner_up = max(others) if others else 0
        if mm_hub > 0: mm_conf = (mm_hub - runner_up) / mm_hub

    # Mid Confidence
    mid_conf = 0.0
    mids = [n for n, r in roles.items() if r == "Middleman"]
    fols = [n for n, r in roles.items() if r == "Follower"]
    if mids and fols:
        avg_mid = sum(authorities[n] for n in mids) / len(mids)
        avg_fol = sum(authorities[n] for n in fols) / len(fols)
        if avg_mid > 0: mid_conf = min((avg_mid - avg_fol) / avg_mid, 1.0)

    system_confidence = (0.6 * mm_conf) + (0.4 * mid_conf)

    # 7. Save Outputs
    # 7. Save Outputs
    # Save CSV
    df = pd.DataFrame({'Hub': hubs, 'Auth': authorities, 'Role': roles})
    df.to_csv(output_scores_csv)

    # --- UPDATED VISUALIZATION BLOCK ---
    if G.number_of_nodes() > 0:
        # Use spring layout with k parameter for better spacing
        pos = nx.spring_layout(G, k=2.5, seed=42, iterations=80)

        # Define colors based on roles
        role_colors_map = { "Mastermind": "red", "Middleman": "orange", "Follower": "skyblue" }
        node_colors = [role_colors_map.get(roles.get(node, "Follower"), "skyblue") for node in G.nodes()]
        edge_weights = [G[u][v]['weight'] for u, v in G.edges()]

        plt.figure(figsize=(16, 12))

        # Draw Nodes
        nx.draw_networkx_nodes(G, pos, node_size=3000, node_color=node_colors, edgecolors='black')

        # Draw Edges (width based on weight)
        nx.draw_networkx_edges(G, pos, width=[w * 2 for w in edge_weights],
                               edge_color='gray', arrowsize=20, node_size=3000)

        # Draw Labels
        nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')

        # Draw Edge Weights
        edge_labels = nx.get_edge_attributes(G, 'weight')
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='darkgreen')

        # Custom Legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', label='Mastermind (Top Hub)', markerfacecolor='red', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Middleman (High Authority)', markerfacecolor='orange', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Follower', markerfacecolor='skyblue', markersize=15)
        ]
        plt.legend(handles=legend_elements, title="Roles", loc="upper right")

        plt.title(f"Influence Network - Dataset {dataset_num}", size=20)
        plt.axis('off')

        # Save to the dynamic path defined earlier
        plt.savefig(output_graph_img)
        plt.close() # Important: Close plot to free memory in loop
    else:
        print("   ⚠️ Graph is empty, skipping visualization.")

    print(f"   ✅ Saved: Graph & CSV. | Confidence: {system_confidence:.2%}")
    return system_confidence


# ==========================================
# 3. MAIN EXECUTION LOOP
# ==========================================

# --- CONFIGURATION ---
# Select modalities here: ['text', 'audio', 'image']
# You can perform combinations like ['image', 'audio'] or just ['text']
selected_modalities = ['text', 'audio']

# Range of datasets (1 to 8)
dataset_range = range(1, 9)
# ---------------------

print("\n" + "="*40)
print(f"🚀 STARTING BATCH PROCESS")
print(f"   Modalities Active: {selected_modalities}")
print("="*40)

results_summary = []

for i in dataset_range:
    score = process_criminal_dataset(i, selected_modalities)
    results_summary.append({
        "Dataset": f"Data {i}",
        "Modality": "+".join(selected_modalities),
        "Confidence Score": f"{score:.2%}"
    })

# --- Final Output ---
print("\n" + "="*40)
print("📊 FINAL BATCH ACCURACY REPORT")
print("="*40)
results_df = pd.DataFrame(results_summary)
print(results_df)
print("="*40)

⏳ Initializing Global Models (this happens once)...


Device set to use cuda:0


   - Loading Whisper...
   - Loading CLIP...
✅ Models Loaded.


🚀 STARTING BATCH PROCESS
   Modalities Active: ['text', 'audio']

--- Processing Dataset 1 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data1_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 74.35%

--- Processing Dataset 2 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data2_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 77.15%

--- Processing Dataset 3 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data3_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 87.74%

--- Processing Dataset 4 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data4_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 81.97%

--- Processing Dataset 5 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data5_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 71.51%

--- Processing Dataset 6 ---
📂 Loading: /cont

Text+image

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from transformers import pipeline, CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util
import warnings
import pandas as pd
import torch
import json
import whisper
from PIL import Image
import os

# --- Initial Setup ---
warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', None)

# ==========================================
# 1. LOAD MODELS (Run Once Global)
# ==========================================
print("⏳ Initializing Global Models (this happens once)...")

# A. NLP Models
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = ["criminal conspiracy", "issuing a command", "normal conversation"]
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# B. Audio Model (Whisper)
print("   - Loading Whisper...")
audio_model = whisper.load_model("base")

# C. Image Model (CLIP)
print("   - Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# D. Define Knowledge Base
crime_image_labels = [
    "a pile of cash money", "a gun or firearm", "drugs or white powder",
    "a secret map or blueprint", "a black duffel bag", "a police car",
    "a normal photo of a person", "a landscape", "a cat or dog"
]

prototypes_by_category = {
    "Threats & Enforcement": [ "Make him understand the consequences.", "Send a clear message.", "Neutralize the threat." ],
    "Financial Transactions": [ "Confirm the payment.", "Launder the money.", "Settle the accounts.", "Look at this cash." ],
    "Commands & Directives": [ "Get the job done now.", "Proceed with the plan.", "Green-light the operation." ],
    "Secrecy & Evasion": [ "Use the secure line.", "Delete this after reading.", "Use the burner phone." ],
    "Acquiring Resources": [ "Acquire the equipment.", "Source the vehicle.", "Here are the weapons." ],
    "Logistics & Transport": [ "Coordinate the pickup.", "The package is delivered.", "Move the merchandise." ],
    "Reporting & Status Updates": [ "Report your status.", "Target is under surveillance.", "Operation complete." ]
}

category_weights = {
    "Commands & Directives": 1.8, "Financial Transactions": 1.6, "Threats & Enforcement": 1.5,
    "Secrecy & Evasion": 1.2, "Logistics & Transport": 0.8, "Acquiring Resources": 0.7, "Reporting & Status Updates": 0.5
}

# Pre-compute prototype embeddings
prototype_embeddings_by_category = {}
for category, sentences in prototypes_by_category.items():
    prototype_embeddings_by_category[category] = embedding_model.encode(sentences, convert_to_tensor=True)

print("✅ Models Loaded.\n")


# ==========================================
# 2. PROCESSING FUNCTION
# ==========================================
def process_criminal_dataset(dataset_num, active_modalities):
    """
    Args:
        dataset_num (int): Number of the dataset (1-8).
        active_modalities (list): List containing strings like ['text', 'audio', 'image'].
    Returns:
        float: The calculated System Confidence Score.
    """

    # 1. Setup Dynamic Paths
    base_path = "/content/drive/MyDrive/Datasets/New Datasets"
    input_file = f"{base_path}/Data/data{dataset_num}_multimodal_final.json"
    output_graph_img = f"{base_path}/Text_Image_Outputs/data{dataset_num}_multimodal.png"
    output_scores_csv = f"{base_path}/Text_Image_Outputs/data{dataset_num}_multimodal.csv"

    print(f"\n--- Processing Dataset {dataset_num} ---")
    print(f"📂 Loading: {input_file}")

    try:
        with open(input_file, 'r') as f:
            message_data = json.load(f)
    except Exception as e:
        print(f"❌ Failed to load dataset {dataset_num}: {e}")
        return 0.0

    directed_edge_weights = {}

    # 2. Analyze Messages
    for msg in message_data:
        sender = msg['sender']
        receiver = msg['receiver']

        text = msg.get('message_text')
        audio_path = msg.get('audio_path')
        image_path = msg.get('image_path')

        # --- A. Process Audio (Only if 'audio' is requested) ---
        if 'audio' in active_modalities and (not text) and audio_path:
            try:
                transcription = audio_model.transcribe(audio_path)
                text = transcription["text"]
            except: pass

        # --- B. Process Image (Only if 'image' is requested) ---
        if 'image' in active_modalities and (not text) and image_path:
            try:
                image = Image.open(image_path)
                inputs = clip_processor(text=crime_image_labels, images=image, return_tensors="pt", padding=True)
                outputs = clip_model(**inputs)
                best_idx = outputs.logits_per_image.softmax(dim=1).argmax().item()
                text = f"User sent a photo of {crime_image_labels[best_idx]}"
            except: pass

        # --- C. Process Text (Only if 'text' is requested) ---
        # Note: We rely on 'text' being present or generated above.
        # If user turned off text analysis for raw text messages, we skip this block.
        if not text: continue
        if 'text' not in active_modalities and not (audio_path or image_path):
            # If it was a raw text message but 'text' modality is OFF
            continue

        # --- Scoring Logic ---
        try:
            # Zero-shot
            result = classifier(text, candidate_labels, multi_label=False)
            score_map = {label: score for label, score in zip(result['labels'], result['scores'])}
            c_score = (score_map.get("criminal conspiracy", 0) * 1.5) + (score_map.get("issuing a command", 0) * 1.0)

            # Semantic Similarity
            msg_emb = embedding_model.encode(text, convert_to_tensor=True)
            sem_score = 0
            for cat, proto_embs in prototype_embeddings_by_category.items():
                best_sim = torch.max(util.cos_sim(msg_emb, proto_embs)).item()
                curr_score = best_sim * category_weights[cat]
                if curr_score > sem_score: sem_score = curr_score

            # Hybrid Score
            final_weight = ((0.3 * c_score) + (0.7 * sem_score)) * 2.0

            edge = (sender, receiver)
            directed_edge_weights[edge] = directed_edge_weights.get(edge, 0) + final_weight

        except: continue

    # 3. Build Graph & HITS
    G = nx.DiGraph()
    for (u, v), w in directed_edge_weights.items():
        if w > 1.2: G.add_edge(u, v, weight=round(w, 2))

    if G.number_of_nodes() == 0:
        print("   ⚠️ Graph empty (no strong evidence found).")
        return 0.0

    hubs, authorities = nx.hits(G, max_iter=1000, normalized=True)

    # 4. Identify Mastermind
    avg_hub = sum(hubs.values()) / len(hubs) if hubs else 0
    potential_mms = [n for n in G.nodes() if hubs[n] > avg_hub * 1.2]
    mastermind = max(potential_mms, key=lambda x: hubs[x]) if potential_mms else None

    # 5. Identify Roles
    avg_auth = sum(authorities.values()) / len(authorities) if authorities else 0
    roles = {}
    for n in G.nodes():
        if n == mastermind: roles[n] = "Mastermind"
        elif authorities[n] > avg_auth * 1.5: roles[n] = "Middleman"
        else: roles[n] = "Follower"

    # 6. Calculate Confidence
    # MM Confidence
    mm_conf = 0.0
    if mastermind:
        mm_hub = hubs[mastermind]
        others = [h for n, h in hubs.items() if n != mastermind]
        runner_up = max(others) if others else 0
        if mm_hub > 0: mm_conf = (mm_hub - runner_up) / mm_hub

    # Mid Confidence
    mid_conf = 0.0
    mids = [n for n, r in roles.items() if r == "Middleman"]
    fols = [n for n, r in roles.items() if r == "Follower"]
    if mids and fols:
        avg_mid = sum(authorities[n] for n in mids) / len(mids)
        avg_fol = sum(authorities[n] for n in fols) / len(fols)
        if avg_mid > 0: mid_conf = min((avg_mid - avg_fol) / avg_mid, 1.0)

    system_confidence = (0.6 * mm_conf) + (0.4 * mid_conf)

    # 7. Save Outputs
    # 7. Save Outputs
    # Save CSV
    df = pd.DataFrame({'Hub': hubs, 'Auth': authorities, 'Role': roles})
    df.to_csv(output_scores_csv)

    # --- UPDATED VISUALIZATION BLOCK ---
    if G.number_of_nodes() > 0:
        # Use spring layout with k parameter for better spacing
        pos = nx.spring_layout(G, k=2.5, seed=42, iterations=80)

        # Define colors based on roles
        role_colors_map = { "Mastermind": "red", "Middleman": "orange", "Follower": "skyblue" }
        node_colors = [role_colors_map.get(roles.get(node, "Follower"), "skyblue") for node in G.nodes()]
        edge_weights = [G[u][v]['weight'] for u, v in G.edges()]

        plt.figure(figsize=(16, 12))

        # Draw Nodes
        nx.draw_networkx_nodes(G, pos, node_size=3000, node_color=node_colors, edgecolors='black')

        # Draw Edges (width based on weight)
        nx.draw_networkx_edges(G, pos, width=[w * 2 for w in edge_weights],
                               edge_color='gray', arrowsize=20, node_size=3000)

        # Draw Labels
        nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')

        # Draw Edge Weights
        edge_labels = nx.get_edge_attributes(G, 'weight')
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='darkgreen')

        # Custom Legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', label='Mastermind (Top Hub)', markerfacecolor='red', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Middleman (High Authority)', markerfacecolor='orange', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Follower', markerfacecolor='skyblue', markersize=15)
        ]
        plt.legend(handles=legend_elements, title="Roles", loc="upper right")

        plt.title(f"Influence Network - Dataset {dataset_num}", size=20)
        plt.axis('off')

        # Save to the dynamic path defined earlier
        plt.savefig(output_graph_img)
        plt.close() # Important: Close plot to free memory in loop
    else:
        print("   ⚠️ Graph is empty, skipping visualization.")

    print(f"   ✅ Saved: Graph & CSV. | Confidence: {system_confidence:.2%}")
    return system_confidence


# ==========================================
# 3. MAIN EXECUTION LOOP
# ==========================================

# --- CONFIGURATION ---
# Select modalities here: ['text', 'audio', 'image']
# You can perform combinations like ['image', 'audio'] or just ['text']
selected_modalities = ['text', 'image']

# Range of datasets (1 to 8)
dataset_range = range(1, 9)
# ---------------------

print("\n" + "="*40)
print(f"🚀 STARTING BATCH PROCESS")
print(f"   Modalities Active: {selected_modalities}")
print("="*40)

results_summary = []

for i in dataset_range:
    score = process_criminal_dataset(i, selected_modalities)
    results_summary.append({
        "Dataset": f"Data {i}",
        "Modality": "+".join(selected_modalities),
        "Confidence Score": f"{score:.2%}"
    })

# --- Final Output ---
print("\n" + "="*40)
print("📊 FINAL BATCH ACCURACY REPORT")
print("="*40)
results_df = pd.DataFrame(results_summary)
print(results_df)
print("="*40)

⏳ Initializing Global Models (this happens once)...


Device set to use cuda:0


   - Loading Whisper...
   - Loading CLIP...
✅ Models Loaded.


🚀 STARTING BATCH PROCESS
   Modalities Active: ['text', 'image']

--- Processing Dataset 1 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data1_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 78.74%

--- Processing Dataset 2 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data2_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 77.91%

--- Processing Dataset 3 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data3_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 83.74%

--- Processing Dataset 4 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data4_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 83.10%

--- Processing Dataset 5 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data5_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 27.99%

--- Processing Dataset 6 ---
📂 Loading: /cont

Image+Audio

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from transformers import pipeline, CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util
import warnings
import pandas as pd
import torch
import json
import whisper
from PIL import Image
import os

# --- Initial Setup ---
warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', None)

# ==========================================
# 1. LOAD MODELS (Run Once Global)
# ==========================================
print("⏳ Initializing Global Models (this happens once)...")

# A. NLP Models
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = ["criminal conspiracy", "issuing a command", "normal conversation"]
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# B. Audio Model (Whisper)
print("   - Loading Whisper...")
audio_model = whisper.load_model("base")

# C. Image Model (CLIP)
print("   - Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# D. Define Knowledge Base
crime_image_labels = [
    "a pile of cash money", "a gun or firearm", "drugs or white powder",
    "a secret map or blueprint", "a black duffel bag", "a police car",
    "a normal photo of a person", "a landscape", "a cat or dog"
]

prototypes_by_category = {
    "Threats & Enforcement": [ "Make him understand the consequences.", "Send a clear message.", "Neutralize the threat." ],
    "Financial Transactions": [ "Confirm the payment.", "Launder the money.", "Settle the accounts.", "Look at this cash." ],
    "Commands & Directives": [ "Get the job done now.", "Proceed with the plan.", "Green-light the operation." ],
    "Secrecy & Evasion": [ "Use the secure line.", "Delete this after reading.", "Use the burner phone." ],
    "Acquiring Resources": [ "Acquire the equipment.", "Source the vehicle.", "Here are the weapons." ],
    "Logistics & Transport": [ "Coordinate the pickup.", "The package is delivered.", "Move the merchandise." ],
    "Reporting & Status Updates": [ "Report your status.", "Target is under surveillance.", "Operation complete." ]
}

category_weights = {
    "Commands & Directives": 1.8, "Financial Transactions": 1.6, "Threats & Enforcement": 1.5,
    "Secrecy & Evasion": 1.2, "Logistics & Transport": 0.8, "Acquiring Resources": 0.7, "Reporting & Status Updates": 0.5
}

# Pre-compute prototype embeddings
prototype_embeddings_by_category = {}
for category, sentences in prototypes_by_category.items():
    prototype_embeddings_by_category[category] = embedding_model.encode(sentences, convert_to_tensor=True)

print("✅ Models Loaded.\n")


# ==========================================
# 2. PROCESSING FUNCTION
# ==========================================
def process_criminal_dataset(dataset_num, active_modalities):
    """
    Args:
        dataset_num (int): Number of the dataset (1-8).
        active_modalities (list): List containing strings like ['text', 'audio', 'image'].
    Returns:
        float: The calculated System Confidence Score.
    """

    # 1. Setup Dynamic Paths
    base_path = "/content/drive/MyDrive/Datasets/New Datasets"
    input_file = f"{base_path}/Data/data{dataset_num}_multimodal_final.json"
    output_graph_img = f"{base_path}/Audio_Image_Outputs/data{dataset_num}_multimodal.png"
    output_scores_csv = f"{base_path}/Audio_Image_Outputs/data{dataset_num}_multimodal.csv"

    print(f"\n--- Processing Dataset {dataset_num} ---")
    print(f"📂 Loading: {input_file}")

    try:
        with open(input_file, 'r') as f:
            message_data = json.load(f)
    except Exception as e:
        print(f"❌ Failed to load dataset {dataset_num}: {e}")
        return 0.0

    directed_edge_weights = {}

    # 2. Analyze Messages
    for msg in message_data:
        sender = msg['sender']
        receiver = msg['receiver']

        text = msg.get('message_text')
        audio_path = msg.get('audio_path')
        image_path = msg.get('image_path')

        # --- A. Process Audio (Only if 'audio' is requested) ---
        if 'audio' in active_modalities and (not text) and audio_path:
            try:
                transcription = audio_model.transcribe(audio_path)
                text = transcription["text"]
            except: pass

        # --- B. Process Image (Only if 'image' is requested) ---
        if 'image' in active_modalities and (not text) and image_path:
            try:
                image = Image.open(image_path)
                inputs = clip_processor(text=crime_image_labels, images=image, return_tensors="pt", padding=True)
                outputs = clip_model(**inputs)
                best_idx = outputs.logits_per_image.softmax(dim=1).argmax().item()
                text = f"User sent a photo of {crime_image_labels[best_idx]}"
            except: pass

        # --- C. Process Text (Only if 'text' is requested) ---
        # Note: We rely on 'text' being present or generated above.
        # If user turned off text analysis for raw text messages, we skip this block.
        if not text: continue
        if 'text' not in active_modalities and not (audio_path or image_path):
            # If it was a raw text message but 'text' modality is OFF
            continue

        # --- Scoring Logic ---
        try:
            # Zero-shot
            result = classifier(text, candidate_labels, multi_label=False)
            score_map = {label: score for label, score in zip(result['labels'], result['scores'])}
            c_score = (score_map.get("criminal conspiracy", 0) * 1.5) + (score_map.get("issuing a command", 0) * 1.0)

            # Semantic Similarity
            msg_emb = embedding_model.encode(text, convert_to_tensor=True)
            sem_score = 0
            for cat, proto_embs in prototype_embeddings_by_category.items():
                best_sim = torch.max(util.cos_sim(msg_emb, proto_embs)).item()
                curr_score = best_sim * category_weights[cat]
                if curr_score > sem_score: sem_score = curr_score

            # Hybrid Score
            final_weight = ((0.3 * c_score) + (0.7 * sem_score)) * 2.0

            edge = (sender, receiver)
            directed_edge_weights[edge] = directed_edge_weights.get(edge, 0) + final_weight

        except: continue

    # 3. Build Graph & HITS
    G = nx.DiGraph()
    for (u, v), w in directed_edge_weights.items():
        if w > 1.2: G.add_edge(u, v, weight=round(w, 2))

    if G.number_of_nodes() == 0:
        print("   ⚠️ Graph empty (no strong evidence found).")
        return 0.0

    hubs, authorities = nx.hits(G, max_iter=1000, normalized=True)

    # 4. Identify Mastermind
    avg_hub = sum(hubs.values()) / len(hubs) if hubs else 0
    potential_mms = [n for n in G.nodes() if hubs[n] > avg_hub * 1.2]
    mastermind = max(potential_mms, key=lambda x: hubs[x]) if potential_mms else None

    # 5. Identify Roles
    avg_auth = sum(authorities.values()) / len(authorities) if authorities else 0
    roles = {}
    for n in G.nodes():
        if n == mastermind: roles[n] = "Mastermind"
        elif authorities[n] > avg_auth * 1.5: roles[n] = "Middleman"
        else: roles[n] = "Follower"

    # 6. Calculate Confidence
    # MM Confidence
    mm_conf = 0.0
    if mastermind:
        mm_hub = hubs[mastermind]
        others = [h for n, h in hubs.items() if n != mastermind]
        runner_up = max(others) if others else 0
        if mm_hub > 0: mm_conf = (mm_hub - runner_up) / mm_hub

    # Mid Confidence
    mid_conf = 0.0
    mids = [n for n, r in roles.items() if r == "Middleman"]
    fols = [n for n, r in roles.items() if r == "Follower"]
    if mids and fols:
        avg_mid = sum(authorities[n] for n in mids) / len(mids)
        avg_fol = sum(authorities[n] for n in fols) / len(fols)
        if avg_mid > 0: mid_conf = min((avg_mid - avg_fol) / avg_mid, 1.0)

    system_confidence = (0.6 * mm_conf) + (0.4 * mid_conf)

    # 7. Save Outputs
    # 7. Save Outputs
    # Save CSV
    df = pd.DataFrame({'Hub': hubs, 'Auth': authorities, 'Role': roles})
    df.to_csv(output_scores_csv)

    # --- UPDATED VISUALIZATION BLOCK ---
    if G.number_of_nodes() > 0:
        # Use spring layout with k parameter for better spacing
        pos = nx.spring_layout(G, k=2.5, seed=42, iterations=80)

        # Define colors based on roles
        role_colors_map = { "Mastermind": "red", "Middleman": "orange", "Follower": "skyblue" }
        node_colors = [role_colors_map.get(roles.get(node, "Follower"), "skyblue") for node in G.nodes()]
        edge_weights = [G[u][v]['weight'] for u, v in G.edges()]

        plt.figure(figsize=(16, 12))

        # Draw Nodes
        nx.draw_networkx_nodes(G, pos, node_size=3000, node_color=node_colors, edgecolors='black')

        # Draw Edges (width based on weight)
        nx.draw_networkx_edges(G, pos, width=[w * 2 for w in edge_weights],
                               edge_color='gray', arrowsize=20, node_size=3000)

        # Draw Labels
        nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')

        # Draw Edge Weights
        edge_labels = nx.get_edge_attributes(G, 'weight')
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='darkgreen')

        # Custom Legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', label='Mastermind (Top Hub)', markerfacecolor='red', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Middleman (High Authority)', markerfacecolor='orange', markersize=15),
            Line2D([0], [0], marker='o', color='w', label='Follower', markerfacecolor='skyblue', markersize=15)
        ]
        plt.legend(handles=legend_elements, title="Roles", loc="upper right")

        plt.title(f"Influence Network - Dataset {dataset_num}", size=20)
        plt.axis('off')

        # Save to the dynamic path defined earlier
        plt.savefig(output_graph_img)
        plt.close() # Important: Close plot to free memory in loop
    else:
        print("   ⚠️ Graph is empty, skipping visualization.")

    print(f"   ✅ Saved: Graph & CSV. | Confidence: {system_confidence:.2%}")
    return system_confidence


# ==========================================
# 3. MAIN EXECUTION LOOP
# ==========================================

# --- CONFIGURATION ---
# Select modalities here: ['text', 'audio', 'image']
# You can perform combinations like ['image', 'audio'] or just ['text']
selected_modalities = ['audio', 'image']

# Range of datasets (1 to 8)
dataset_range = range(1, 9)
# ---------------------

print("\n" + "="*40)
print(f"🚀 STARTING BATCH PROCESS")
print(f"   Modalities Active: {selected_modalities}")
print("="*40)

results_summary = []

for i in dataset_range:
    score = process_criminal_dataset(i, selected_modalities)
    results_summary.append({
        "Dataset": f"Data {i}",
        "Modality": "+".join(selected_modalities),
        "Confidence Score": f"{score:.2%}"
    })

# --- Final Output ---
print("\n" + "="*40)
print("📊 FINAL BATCH ACCURACY REPORT")
print("="*40)
results_df = pd.DataFrame(results_summary)
print(results_df)
print("="*40)

⏳ Initializing Global Models (this happens once)...


Device set to use cuda:0


   - Loading Whisper...
   - Loading CLIP...
✅ Models Loaded.


🚀 STARTING BATCH PROCESS
   Modalities Active: ['audio', 'image']

--- Processing Dataset 1 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data1_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 100.00%

--- Processing Dataset 2 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data2_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 72.13%

--- Processing Dataset 3 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data3_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 84.45%

--- Processing Dataset 4 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data4_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 67.45%

--- Processing Dataset 5 ---
📂 Loading: /content/drive/MyDrive/Datasets/New Datasets/Data/data5_multimodal_final.json
   ✅ Saved: Graph & CSV. | Confidence: 78.26%

--- Processing Dataset 6 ---
📂 Loading: /co